# Robot Dynamics Identification Pipeline — Results Visualization

本 Notebook 将辨识流水线的输出结果可视化，涵盖：
- 全局误差（RMSE / MAE）随流程阶段的变化
- 每个关节在三个阶段的误差对比
- 线性补偿 vs MLP 补偿的改善率
- 5-seed 稳定性分析
- RMSE 热力图
- 并行坐标折线图

---
**环境要求**：`pip install matplotlib numpy`

In [3]:
from plot_results_support import load_plot_context

globals().update(load_plot_context(save_figures=False))

print('=' * 72)
print('当前可视化结果摘要')
print('=' * 72)
print(f'结果时间: {GENERATED_AT}')
print(f'机器人模型: {ROBOT_NAME}')
print(f'数据来源: {DATA_SOURCE}')
print(f'结果文件: {VIS_FILE}')
print(f'图片保存: {SAVE_FIGURES}  (False=只显示不保存, True=覆盖保存到 {FIGURE_DIR})')
print(f"Identification test RMSE: {payload['identification']['test']['global_rmse']:.6f} N·m")
print(f"Linear test RMSE:         {payload['linear']['test']['global_rmse']:.6f} N·m")
print(f"MLP test RMSE:            {payload['mlp']['test']['global_rmse']:.6f} N·m")
print(f"Linear improvement:       {payload['linear']['test']['improvement_percent']:.2f}%")
print(f"MLP improvement:          {payload['mlp']['test']['improvement_percent']:.2f}%")


✅ 数据加载完成


---
## Fig 1 — 流程各阶段全局误差（RMSE / MAE）

In [1]:
fig, ax = plt.subplots(figsize=(8, 4.5))

x   = np.arange(3)
w   = 0.32

bars1 = ax.bar(x - w/2, rmse_global, w, color=C_BLUE,   label='RMSE', zorder=3)
bars2 = ax.bar(x + w/2, mae_global,  w, color=C_ORANGE, label='MAE',  zorder=3)

# 数值标注
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.06,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, color=C_BLUE)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.06,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, color=C_ORANGE)

# 改善率箭头：这里直接用主程序输出的最新 improvement 指标。
improvements = global_improvements
for i, pct in enumerate(improvements):
    y0 = rmse_global[i]   + 0.45
    y1 = rmse_global[i+1] + 0.45
    ax.annotate('', xy=(i+1, y1), xytext=(i, y0),
                arrowprops=dict(arrowstyle='->', color=C_GREEN, lw=1.4))
    ax.text(i + 0.5, max(y0, y1) + 0.12, f'−{pct:.1f}%',
            ha='center', fontsize=9, color=C_GREEN, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(STAGES, fontsize=11)
ax.set_ylabel('Torque error (N·m)')
ax.set_ylim(0, max(rmse_global) * 1.35)
ax.set_title('Global test-set error across pipeline stages', fontsize=13, fontweight='normal')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('fig1_pipeline_overview.png', dpi=150)
plt.show()

NameError: name 'plt' is not defined

---
## Fig 2 — 每关节 RMSE：三阶段对比

In [2]:
fig, ax = plt.subplots(figsize=(10, 5))

x, w = np.arange(7), 0.25

ax.bar(x - w, rmse_ident,  w, color=C_BLUE,   label='Identification', zorder=3)
ax.bar(x,     rmse_linear, w, color=C_ORANGE, label='+ Linear comp.', zorder=3)
ax.bar(x + w, rmse_mlp,    w, color=C_GREEN,  label='+ MLP comp.',    zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(JOINTS)
ax.set_ylabel('RMSE (N·m)')
ax.set_ylim(0, rmse_ident.max() * 1.22)
ax.set_title('Per-joint RMSE on test set — three pipeline stages', fontweight='normal')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('fig2_per_joint_rmse.png', dpi=150)
plt.show()

NameError: name 'plt' is not defined

---
## Fig 3 — 每关节 MAE：三阶段对比

In [3]:
fig, ax = plt.subplots(figsize=(10, 5))

x, w = np.arange(7), 0.25

ax.bar(x - w, mae_ident,  w, color=C_BLUE,   label='Identification', zorder=3)
ax.bar(x,     mae_linear, w, color=C_ORANGE, label='+ Linear comp.', zorder=3)
ax.bar(x + w, mae_mlp,    w, color=C_GREEN,  label='+ MLP comp.',    zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(JOINTS)
ax.set_ylabel('MAE (N·m)')
ax.set_ylim(0, mae_ident.max() * 1.22)
ax.set_title('Per-joint MAE on test set — three pipeline stages', fontweight='normal')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('fig3_per_joint_mae.png', dpi=150)
plt.show()

NameError: name 'plt' is not defined

---
## Fig 4 — 每关节改善率（%）：线性 vs MLP

In [4]:
fig, ax = plt.subplots(figsize=(10, 5))

x, w = np.arange(7), 0.32

ax.bar(x - w/2, impr_linear, w, color=C_ORANGE, label='Linear compensator', zorder=3)
ax.bar(x + w/2, impr_mlp,    w, color=C_GREEN,  label='MLP compensator',    zorder=3)

# 50% 参考线
ax.axhline(50, color=C_GRAY, linestyle='--', linewidth=1, zorder=2)
ax.text(6.6, 51.5, '50%', color=C_GRAY, fontsize=9)

# 数值标注
for i in range(7):
    ax.text(i - w/2, impr_linear[i] + 1.2, f'{impr_linear[i]:.0f}%',
            ha='center', fontsize=8, color='#7A4500')
    ax.text(i + w/2, impr_mlp[i]    + 1.2, f'{impr_mlp[i]:.0f}%',
            ha='center', fontsize=8, color='#1A6000')

ax.set_xticks(x)
ax.set_xticklabels(JOINTS)
ax.set_ylabel('RMSE improvement over identification (%)')
ax.set_ylim(0, 108)
ax.set_title('Per-joint RMSE improvement on test set', fontweight='normal')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('fig4_per_joint_improvement.png', dpi=150)
plt.show()

NameError: name 'plt' is not defined

---
## Fig 5 — 5-seed 稳定性：Train / Val / Test RMSE

In [5]:
fig, ax = plt.subplots(figsize=(6, 4))

splits    = ['Train', 'Val', 'Test']
rmse_mean = stability_rmse_mean
rmse_std  = stability_rmse_std

ax.errorbar(range(3), rmse_mean, yerr=rmse_std,
            fmt='o-', color=C_BLUE, markerfacecolor=C_BLUE,
            linewidth=2, markersize=8, capsize=8, capthick=1.5, zorder=3)

# 填充 ±1σ 带
ax.fill_between(range(3), rmse_mean - rmse_std, rmse_mean + rmse_std,
                alpha=0.15, color=C_BLUE)

for i, (m, s) in enumerate(zip(rmse_mean, rmse_std)):
    ax.text(i, m + s + 0.008, f'{m:.3f}±{s:.3f}',
            ha='center', fontsize=9, color=C_BLUE)

ax.set_xticks(range(3))
ax.set_xticklabels(splits)
ax.set_ylabel('RMSE (N·m)')
ax.set_ylim(4.20, 4.40)
ax.set_title('Identification RMSE — stability over 5 seeds', fontweight='normal')

plt.tight_layout()
plt.savefig('fig5_stability.png', dpi=150)
plt.show()

NameError: name 'plt' is not defined

---
## Fig 6 — RMSE 热力图（关节 × 流程阶段）

In [6]:
fig, ax = plt.subplots(figsize=(7, 4))

# 矩阵：行=关节，列=阶段
heatmap = np.vstack([rmse_ident, rmse_linear, rmse_mlp]).T  # (7, 3)

im = ax.imshow(heatmap, cmap='YlOrRd', aspect='auto')
cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label('RMSE (N·m)', fontsize=11)

ax.set_xticks(range(3))
ax.set_xticklabels(STAGES, fontsize=10)
ax.set_yticks(range(7))
ax.set_yticklabels(JOINTS)
ax.set_title('Per-joint RMSE heat-map (test set)', fontweight='normal')

# 单元格数值
threshold = heatmap.max() * 0.55
for r in range(7):
    for c in range(3):
        val = heatmap[r, c]
        color = 'white' if val > threshold else 'black'
        ax.text(c, r, f'{val:.2f}', ha='center', va='center',
                fontsize=9, color=color)

plt.tight_layout()
plt.savefig('fig6_heatmap.png', dpi=150)
plt.show()

NameError: name 'plt' is not defined

---
## Fig 7 — 并行坐标折线图（J1–J6 RMSE 轮廓）

In [7]:
fig, ax = plt.subplots(figsize=(9, 4.5))

xax = np.arange(6)
mi  = rmse_ident[:6]
ml  = rmse_linear[:6]
mm  = rmse_mlp[:6]

ax.plot(xax, mi, 'o-', color=C_BLUE,   lw=2, ms=7, label='Identification',  zorder=4)
ax.plot(xax, ml, 's-', color=C_ORANGE, lw=2, ms=7, label='+ Linear comp.',  zorder=4)
ax.plot(xax, mm, '^-', color=C_GREEN,  lw=2, ms=7, label='+ MLP comp.',     zorder=4)

# 填充辨识到MLP的改善区
ax.fill_between(xax, mm, mi, alpha=0.08, color=C_GREEN, label='MLP improvement area')

# 数值标注（MLP）
for i, v in enumerate(mm):
    ax.text(i, v + 0.12, f'{v:.2f}', ha='center', fontsize=8, color=C_GREEN)

ax.set_xticks(xax)
ax.set_xticklabels(JOINTS[:6])
ax.set_ylabel('RMSE (N·m)')
ax.set_title('Per-joint RMSE profile across pipeline (J1–J6)', fontweight='normal')
ax.legend(fontsize=10, loc='upper right')

plt.tight_layout()
plt.savefig('fig7_parallel_coords.png', dpi=150)
plt.show()

NameError: name 'plt' is not defined

---
## Fig 8 — 综合总览（2×2 子图）

In [8]:
fig = plt.figure(figsize=(14, 10))
gs  = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.32)

# ── 子图 A：全局误差 ──────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
x, w = np.arange(3), 0.32
ax_a.bar(x - w/2, rmse_global, w, color=C_BLUE,   label='RMSE', zorder=3)
ax_a.bar(x + w/2, mae_global,  w, color=C_ORANGE, label='MAE',  zorder=3)
ax_a.set_xticks(x)
ax_a.set_xticklabels(STAGES, fontsize=9)
ax_a.set_ylabel('Torque error (N·m)')
ax_a.set_title('(A) Global error by stage', fontweight='normal')
ax_a.legend(fontsize=9)
ax_a.set_ylim(0, rmse_global.max() * 1.3)
ax_a.grid(True, alpha=0.35, linestyle='--')
ax_a.spines['top'].set_visible(False)
ax_a.spines['right'].set_visible(False)

# ── 子图 B：改善率 ────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
x7, w7 = np.arange(7), 0.32
ax_b.bar(x7 - w7/2, impr_linear, w7, color=C_ORANGE, label='Linear', zorder=3)
ax_b.bar(x7 + w7/2, impr_mlp,    w7, color=C_GREEN,  label='MLP',    zorder=3)
ax_b.axhline(50, color=C_GRAY, linestyle='--', lw=1)
ax_b.set_xticks(x7)
ax_b.set_xticklabels(JOINTS, fontsize=9)
ax_b.set_ylabel('RMSE improvement (%)')
ax_b.set_ylim(0, 110)
ax_b.set_title('(B) Per-joint improvement', fontweight='normal')
ax_b.legend(fontsize=9)
ax_b.grid(True, alpha=0.35, linestyle='--')
ax_b.spines['top'].set_visible(False)
ax_b.spines['right'].set_visible(False)

# ── 子图 C：热力图 ────────────────────────────────
ax_c = fig.add_subplot(gs[1, 0])
heatmap = np.vstack([rmse_ident, rmse_linear, rmse_mlp]).T
im = ax_c.imshow(heatmap, cmap='YlOrRd', aspect='auto')
fig.colorbar(im, ax=ax_c, pad=0.02).set_label('RMSE (N·m)', fontsize=9)
ax_c.set_xticks(range(3))
ax_c.set_xticklabels(['Ident.', '+Linear', '+MLP'], fontsize=9)
ax_c.set_yticks(range(7))
ax_c.set_yticklabels(JOINTS)
ax_c.set_title('(C) RMSE heat-map', fontweight='normal')
threshold = heatmap.max() * 0.55
for r in range(7):
    for c in range(3):
        v = heatmap[r, c]
        ax_c.text(c, r, f'{v:.2f}', ha='center', va='center',
                  fontsize=8, color='white' if v > threshold else 'black')

# ── 子图 D：折线轮廓 ──────────────────────────────
ax_d = fig.add_subplot(gs[1, 1])
xax6 = np.arange(6)
ax_d.plot(xax6, rmse_ident[:6],  'o-', color=C_BLUE,   lw=2, ms=6, label='Identification')
ax_d.plot(xax6, rmse_linear[:6], 's-', color=C_ORANGE, lw=2, ms=6, label='+ Linear')
ax_d.plot(xax6, rmse_mlp[:6],    '^-', color=C_GREEN,  lw=2, ms=6, label='+ MLP')
ax_d.fill_between(xax6, rmse_mlp[:6], rmse_ident[:6], alpha=0.08, color=C_GREEN)
ax_d.set_xticks(xax6)
ax_d.set_xticklabels(JOINTS[:6], fontsize=9)
ax_d.set_ylabel('RMSE (N·m)')
ax_d.set_title('(D) RMSE profile J1–J6', fontweight='normal')
ax_d.legend(fontsize=9)
ax_d.grid(True, alpha=0.35, linestyle='--')
ax_d.spines['top'].set_visible(False)
ax_d.spines['right'].set_visible(False)

fig.suptitle('Robot Dynamics Identification — Pipeline Results Summary',
             fontsize=14, fontweight='normal', y=1.01)

plt.savefig('fig8_summary_2x2.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 所有图表已保存')

NameError: name 'plt' is not defined